# Paso 00 — Preparar demanda de los escalones (verificación del simulador)

Genera y guarda `grupos_escalon1.csv` (franja mañana, pocos grupos, para
verificar a ojo), `grupos_escalon2.csv` (día completo, más denso, para
métricas agregadas) y `grupos_semana.csv` de escalón 3 (semana completa,
7 días, entre semana + fin de semana) -- ver `docs/especificacion_simulador_rl.md`
y `simulacion/README.md`.

**No se redefine el generador de demanda.** Se reusa `demand/src/llegadas.py`
tal cual (misma intensidad O-D ya calculada en `demand/output/matriz_intensidad_od.csv`,
mismo patrón gravedad+SSB+Poisson), solo con un `porcentaje_poblacion_dia`
más bajo -- una copia del config de demanda, nunca se toca
`demand/config/instance.yaml`, que sigue siendo el oficial al 10%.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
DEMAND_DIR = (BASE_DIR / "../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas

cfg_sim = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()

bb_cfg_path = (DEMAND_DIR / cfg_demand["fuentes_externas"]["nodos_config"]).resolve()
cfg_bb = yaml.safe_load(open(bb_cfg_path, encoding="utf-8"))
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Poblacion total 4 zonas:", poblacion_total_zonas)
print("porcentaje_poblacion_dia oficial (demand/):", cfg_demand["demanda"]["porcentaje_poblacion_dia"])
print("Semilla:", cfg_sim["semilla"])


Poblacion total 4 zonas: 79943
porcentaje_poblacion_dia oficial (demand/): 0.1
Semilla: 42


## Generar y filtrar cada escalón

In [2]:
def generar_grupos_escalon(nombre_escalon: str) -> pd.DataFrame:
    cfg_escalon = cfg_sim["escalones"][nombre_escalon]
    cfg_mod = dict(cfg_demand)
    cfg_mod["demanda"] = dict(cfg_demand["demanda"])
    cfg_mod["demanda"]["porcentaje_poblacion_dia"] = cfg_escalon["porcentaje_poblacion_dia"]

    rng = np.random.default_rng(cfg_sim["semilla"])
    grupos = demand_llegadas.generar_llegadas_dia(
        cfg_mod, intensidad_od, conexiones_fuertes, poblacion_total_zonas,
        es_fin_de_semana=False, rng=rng, dia_id=nombre_escalon,
    )
    hora_ini, hora_fin = cfg_escalon["horas"]
    grupos = grupos[(grupos["hora"] >= hora_ini) & (grupos["hora"] < hora_fin)].reset_index(drop=True)
    return grupos


grupos_escalon1 = generar_grupos_escalon("escalon_1")
grupos_escalon2 = generar_grupos_escalon("escalon_2")

print(f"Escalon 1 (manana): {len(grupos_escalon1)} grupos, {grupos_escalon1['tamano_grupo'].sum()} personas")
print(f"Escalon 2 (dia completo): {len(grupos_escalon2)} grupos, {grupos_escalon2['tamano_grupo'].sum()} personas")
grupos_escalon1.head(10)


Escalon 1 (manana): 23 grupos, 202 personas


Escalon 2 (dia completo): 133 grupos, 1106 personas


,grupo_id,dia,franja,hora,minuto_dia,origen,destino,tamano_grupo,espera_maxima_min,conexion_fuerte,fin_de_semana
0,escalon_1_00013,escalon_1,manana,6,366.87,laksevag,bryggen,10,13.0,True,False
1,escalon_1_00012,escalon_1,manana,6,368.39,laksevag,bryggen,9,15.6,True,False
2,escalon_1_00020,escalon_1,manana,6,378.36,sandviken,bryggen,6,17.6,True,False
3,escalon_1_00011,escalon_1,manana,6,394.12,laksevag,bryggen,8,15.5,True,False
4,escalon_1_00021,escalon_1,manana,6,394.75,sandviken,bryggen,11,17.2,True,False
5,escalon_1_00005,escalon_1,manana,6,397.90,kleppesto,bryggen,8,18.9,True,False
6,escalon_1_00014,escalon_1,manana,6,400.10,laksevag,bryggen,9,10.3,True,False
7,escalon_1_00019,escalon_1,manana,6,401.98,sandviken,kleppesto,7,34.7,False,False
8,escalon_1_00004,escalon_1,manana,6,409.66,kleppesto,bryggen,10,19.7,True,False
9,escalon_1_00001,escalon_1,manana,6,411.52,kleppesto,laksevag,10,25.9,False,False


In [3]:
def generar_grupos_semana_escalon(nombre_escalon: str) -> pd.DataFrame:
    """Igual que `generar_grupos_escalon`, pero para una semana completa --
    usa `generar_llegadas_semana` (7 días, una semilla hija por día vía
    `numpy.random.SeedSequence`, entre semana/fin de semana ya resuelto
    adentro, columna `dia` 0=lunes..6=domingo) en vez de `generar_llegadas_dia`.
    """
    cfg_escalon = cfg_sim["escalones"][nombre_escalon]
    cfg_mod = dict(cfg_demand)
    cfg_mod["demanda"] = dict(cfg_demand["demanda"])
    cfg_mod["demanda"]["porcentaje_poblacion_dia"] = cfg_escalon["porcentaje_poblacion_dia"]

    grupos = demand_llegadas.generar_llegadas_semana(
        cfg_mod, intensidad_od, conexiones_fuertes, poblacion_total_zonas, semilla=cfg_sim["semilla"],
    )
    hora_ini, hora_fin = cfg_escalon["horas"]
    grupos = grupos[(grupos["hora"] >= hora_ini) & (grupos["hora"] < hora_fin)].reset_index(drop=True)
    return grupos


grupos_escalon3 = generar_grupos_semana_escalon("escalon_3")
print(f"Escalon 3 (semana): {len(grupos_escalon3)} grupos, {grupos_escalon3['tamano_grupo'].sum()} personas")
grupos_escalon3.groupby("dia")["tamano_grupo"].agg(["count", "sum"]).rename(
    columns={"count": "grupos", "sum": "personas"}
)


Escalon 3 (semana): 620 grupos, 5429 personas


,grupos,personas
dia,,
0,113,987
1,121,1013
2,87,756
3,121,1121
4,105,902
5,35,326
6,38,324


In [4]:
(OUTPUT_DIR / "escalon1").mkdir(exist_ok=True)
(OUTPUT_DIR / "escalon2").mkdir(exist_ok=True)
(OUTPUT_DIR / "escalon3").mkdir(exist_ok=True)
grupos_escalon1.to_csv(OUTPUT_DIR / "escalon1" / "grupos.csv", index=False)
grupos_escalon2.to_csv(OUTPUT_DIR / "escalon2" / "grupos.csv", index=False)
grupos_escalon3.to_csv(OUTPUT_DIR / "escalon3" / "grupos_semana.csv", index=False)
print("Guardados:", OUTPUT_DIR / "escalon1" / "grupos.csv,", OUTPUT_DIR / "escalon2" / "grupos.csv,",
      "y", OUTPUT_DIR / "escalon3" / "grupos_semana.csv")


Guardados: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\simulacion\output\escalon1\grupos.csv, C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\simulacion\output\escalon2\grupos.csv, y C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\simulacion\output\escalon3\grupos_semana.csv


## Distribución por par O-D (chequeo de sanidad)

Confirma que el patrón (quién va a dónde) sigue siendo el mismo que en
`demand/` -- solo cambió la escala, no la forma.


In [5]:
grupos_escalon1.groupby(["origen", "destino"])["tamano_grupo"].sum().sort_values(ascending=False)


origen     destino  
kleppesto  bryggen      55
laksevag   bryggen      47
kleppesto  laksevag     30
sandviken  bryggen      29
bryggen    laksevag     16
           kleppesto    11
kleppesto  sandviken     7
sandviken  kleppesto     7
Name: tamano_grupo, dtype: int64